# 03 – Analyse: zorgt regen voor minder bikeshare-gebruik? (dinsdagmiddag)
Opbouw:
1. Lijngrafiek ritten + neerslag (rubric **Lijngrafiek**)
2. Droog vs. regen (boxplot)
3. Correlatie
4. **Eerlijke vergelijking binnen dezelfde maand en hetzelfde type dag** (corrigeren voor seizoen)
5. Werkdag vs. weekend, ritduur, uurpatroon
6. Eerste regressie (opstap naar het voorspelmodel van woensdag)

Elke grafiek krijgt een titel met de **conclusie**. Vul die zelf aan zodra je de uitkomst ziet.

In [ ]:
# Load the daily and hourly tables; drop days without a precipitation/temperature measurement
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

PROC = Path("data/processed")
daily = pd.read_parquet(PROC / "daily.parquet")
hourly = pd.read_parquet(PROC / "hourly.parquet")
n_before = len(daily)
daily = daily.dropna(subset=["precip_mm", "temp_mean"])
CATS = ["droog (<0,2 mm)", "licht (0,2–2)", "matig (2–5)", "zwaar (>5)"]
daily["rain_cat"] = pd.Categorical(daily["rain_cat"], CATS, ordered=True)
print(f"{len(daily)} dagen met weerdata ({n_before - len(daily)} dagen zonder neerslag/temperatuur weggelaten)")

## 1. Lijngrafiek: ritten per dag en neerslag

In [ ]:
# Line chart: rides per day (raw + 7-day average) with precipitation as bars on a second axis
fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.plot(daily["date"], daily["rides"], lw=.7, alpha=.5, label="ritten per dag")
ax1.plot(daily["date"], daily["rides"].rolling(7, center=True).mean(), lw=2, label="7-daags gemiddelde")
ax1.set_ylabel("ritten per dag")
ax2 = ax1.twinx()
ax2.bar(daily["date"], daily["precip_mm"], color="tab:blue", alpha=.35, width=1)
ax2.set_ylabel("neerslag (mm)"); ax2.set_ylim(0, daily["precip_mm"].max() * 3)  # keep bars at the bottom
ax1.legend(loc="upper left")
ax1.set_title("Ritten per dag vs. neerslag – dips vallen vaak samen met regendagen")
plt.show()

## 2. Droog vs. regen

In [ ]:
# Boxplot: distribution of daily rides per rain category
fig, ax = plt.subplots(figsize=(8, 4))
data = [daily.loc[daily["rain_cat"] == c, "rides"] for c in CATS]
ax.boxplot(data)
ax.set_xticks(range(1, len(CATS) + 1), [f"{c}\n(n={len(d)})" for c, d in zip(CATS, data)])
ax.set_ylabel("ritten per dag")
ax.set_title("Hoe natter de dag, hoe minder ritten")
plt.show()
daily.groupby("rain_cat", observed=True)["rides"].agg(["count", "mean", "median"]).round(0)

## 3. Correlatie

In [ ]:
# Scatter plots and correlation of precipitation and temperature with the number of rides
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(daily["precip_mm"], daily["rides"], alpha=.4)
ax[0].set_xlabel("neerslag (mm)"); ax[0].set_ylabel("ritten"); ax[0].set_title("Neerslag vs. ritten")
ax[1].scatter(daily["temp_mean"], daily["rides"], alpha=.4, c=daily["precip_mm"] > 0.2, cmap="coolwarm")
ax[1].set_xlabel("gem. temperatuur (°C)"); ax[1].set_title("Temperatuur vs. ritten (rood = regendag)")
plt.show()

for col in ["precip_mm", "temp_mean"]:
    r, p = stats.pearsonr(daily[col], daily["rides"])
    rho, p2 = stats.spearmanr(daily[col], daily["rides"])
    print(f"{col:<10} Pearson r = {r:+.2f} (p={p:.3g}) | Spearman ρ = {rho:+.2f} (p={p2:.3g})")

## 4. ⭐ Eerlijke vergelijking: corrigeren voor seizoen en type dag
Regen valt vaker in herfst/winter, en dan fietst men sowieso minder. Een simpele vergelijking
droog vs. regen **overschat** dus het effect van regen.

**Aanpak:** voor elke dag vergelijken we het aantal ritten met de **mediaan van de droge dagen in dezelfde
maand en van hetzelfde type dag** (werkdag of weekend). Zo meten we alleen het effect van regen.

In [ ]:
# Fair comparison: each day vs. the median of dry days in the same month and of the same day type
ref = (daily[daily["rain_cat"] == CATS[0]].groupby(["month", "day_type"])["rides"].median().rename("ref_dry"))
daily = daily.merge(ref, on=["month", "day_type"], how="left")
daily["pct_vs_dry"] = (daily["rides"] / daily["ref_dry"] - 1) * 100

effect = daily.groupby("rain_cat", observed=True)["pct_vs_dry"].agg(["count", "mean", "median"]).round(1)
effect

In [ ]:
# Bar chart of the season-corrected effect per rain category
fig, ax = plt.subplots(figsize=(8, 4))
e = effect["mean"].reindex(CATS)
ax.bar(e.index.astype(str), e.values, color=["grey", "#9ecae1", "#4292c6", "#08519c"])
ax.axhline(0, color="black", lw=.8)
ax.set_ylabel("% ritten t.o.v. droge dag\n(zelfde maand & type dag)")
ax.set_title("Gecorrigeerd voor seizoen: zware regen kost ±…% ritten")   # ← fill in
plt.show()

## 5. Wie laat de fiets staan? Werkdag vs. weekend, ritduur en uurpatroon

In [ ]:
# Does rain affect working days and weekends differently?
pv = (daily.pivot_table(index="rain_cat", columns="day_type", values="pct_vs_dry", aggfunc="mean", observed=True)
      .reindex(CATS))
pv.plot.bar(figsize=(8, 4), rot=0)
plt.axhline(0, color="black", lw=.8)
plt.ylabel("% t.o.v. droge dag")
plt.title("Regen raakt recreatief (weekend) gebruik harder dan woon-werk?")  # ← check!
plt.show()
pv.round(1)

In [ ]:
# Do people make shorter rides when it rains?
daily.groupby("rain_cat", observed=True)[["duration_mean", "duration_median"]].mean().round(1)

In [ ]:
# Average rides per hour: dry vs. rainy days, for working days and weekends
h = hourly.dropna(subset=["rain_cat"]).copy()
h["is_rainy"] = np.where(h["rain_cat"].astype(str).str.startswith("droog"), "droog", "regen")
profile = h.groupby(["day_type", "is_rainy", "hour"])["rides"].mean().unstack([0, 1])
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for a, dt in zip(ax, ["werkdag", "weekend/feestdag"]):
    if dt in profile.columns.get_level_values(0):
        profile[dt].plot(ax=a)
    a.set_title(dt); a.set_xlabel("uur"); a.set_ylabel("gem. ritten per uur")
fig.suptitle("Uurpatroon droge vs. regendagen")
plt.show()

## 6. Eerste regressie (preview voor het model)
Lineaire regressie met controle voor temperatuur, weekdag, maand en feestdag.
Met `log(rides)` kun je de coëfficiënt lezen als **procentuele verandering per mm regen**.

In [ ]:
# Linear regression on log(rides): coefficient ≈ % change in rides per mm of rain, controlling for temperature, weekday, month and holidays
m = smf.ols("np.log(rides) ~ precip_mm + temp_mean + C(weekday) + C(month) + is_holiday", data=daily).fit()
b = m.params["precip_mm"]
lo, hi = m.conf_int().loc["precip_mm"]
print(f"Elke mm regen ≈ {(np.exp(b) - 1) * 100:+.1f}% ritten "
      f"(95%-BI {(np.exp(lo) - 1) * 100:+.1f}% t/m {(np.exp(hi) - 1) * 100:+.1f}%), R² = {m.rsquared:.2f}")
print(m.summary().tables[1])

## Conclusie (concept)
- **Hoofdvraag:** …
- **Grootte van het effect:** … (zie stap 4 en 6)
- **Werkdag vs. weekend:** …
- **Kanttekeningen:** correlatie ≠ causaliteit, het weer is gemeten per dag (niet per uur), er is één weerstation
  voor heel Londen, en dagen zonder neerslagmeting zijn weggelaten.

➡️ Woensdag: subvraag (kaart, waar minder gebruik en waarom), voorspelmodel en het Streamlit-dashboard.